In [ ]:
import torch
import torch.nn as nn

class UNetLite(nn.Module):
    def __init__(self):
        super(UNetLite, self).__init__()

        # ---- ENCODER ----
        # Conv2d(in=1, out=8, kernel=3, stride=1, padding=1)
        # Input:  1 x 32 x 32
        # Output: 8 x 32 x 32  → (32 + 2*1 - 3)/1 + 1 = 32 ✓
        self.enc1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, stride=1, padding=1),
            nn.ReLU()
        )

        # MaxPool2d(kernel=2, stride=2, padding=0)
        # Input:  8 x 32 x 32
        # Output: 8 x 16 x 16  → (32 + 0 - 2)/2 + 1 = 16 ✓
        self.pool1 = nn.Sequential(
            nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        )

        # ---- BOTTLENECK ----
        # Conv2d(in=8, out=16, kernel=3, stride=1, padding=1)
        # Input:  8  x 16 x 16
        # Output: 16 x 16 x 16  → (16 + 2*1 - 3)/1 + 1 = 16 ✓
        self.bottleneck = nn.Sequential(
            nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, stride=1, padding=1),
            nn.ReLU()
        )

        # ---- DECODER ----
        # ConvTranspose2d(in=16, out=8, kernel=2, stride=2, padding=0)
        # Input:  16 x 16 x 16
        # Output: 8  x 32 x 32  → (16-1)*2 - 0 + 2 = 32 ✓
        self.up1 = nn.Sequential(
            nn.ConvTranspose2d(in_channels=16, out_channels=8, kernel_size=2, stride=2, padding=0)
        )

        # After Cat1: channels = 8 (from up1) + 8 (from enc1 skip) = 16
        # Conv2d(in=16, out=8, kernel=3, stride=1, padding=1)
        # Input:  16 x 32 x 32
        # Output: 8  x 32 x 32  → (32 + 2*1 - 3)/1 + 1 = 32 ✓
        self.dec1 = nn.Sequential(
            nn.Conv2d(in_channels=16, out_channels=8, kernel_size=3, stride=1, padding=1),
            nn.ReLU()
        )

        # ---- FINAL ----
        # Conv2d(in=8, out=1, kernel=1, stride=1, padding=0)
        # Input:  8 x 32 x 32
        # Output: 1 x 32 x 32  → (32 + 0 - 1)/1 + 1 = 32 ✓
        self.final = nn.Sequential(
            nn.Conv2d(in_channels=8, out_channels=1, kernel_size=1, stride=1, padding=0)
        )

    def forward(self, x):
        # x shape: [batch, 1, 32, 32]

        # Encoder
        enc1_out = self.enc1(x)       # [batch, 8, 32, 32]
        pool1_out = self.pool1(enc1_out)  # [batch, 8, 16, 16]

        # Bottleneck
        bottle_out = self.bottleneck(pool1_out)  # [batch, 16, 16, 16]

        # Decoder - upsample
        up1_out = self.up1(bottle_out)           # [batch, 8, 32, 32]

        # Skip connection: concatenate enc1 output with up1 output along channel dim
        cat1_out = torch.cat([up1_out, enc1_out], dim=1)  # [batch, 16, 32, 32]

        # Decoder - refine
        dec1_out = self.dec1(cat1_out)           # [batch, 8, 32, 32]

        # Final output
        out = self.final(dec1_out)               # [batch, 1, 32, 32]

        return out


# ---- Quick sanity check ----
if __name__ == "__main__":
    model = UNetLite()
    
    # Fake input: batch of 2 grayscale 32x32 images
    x = torch.randn(2, 1, 32, 32)
    output = model(x)
    
    print("Input shape: ", x.shape)       # [2, 1, 32, 32]
    print("Output shape:", output.shape)  # [2, 1, 32, 32]
    print("Parameter count:", sum(p.numel() for p in model.parameters()))
```

---

## What Happens at Each Line in `forward()`
```
x           →  [2,  1, 32, 32]   raw input
enc1_out    →  [2,  8, 32, 32]   saved! will be used in skip connection
pool1_out   →  [2,  8, 16, 16]   spatial halved
bottle_out  →  [2, 16, 16, 16]   channels doubled
up1_out     →  [2,  8, 32, 32]   spatial restored
cat1_out    →  [2, 16, 32, 32]   ← skip fires here (8+8 channels)
dec1_out    →  [2,  8, 32, 32]   refined
out         →  [2,  1, 32, 32]   final output ✓